In [68]:
import torch.nn.functional as F

## 1. RNN-Based Text Generation

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import re
import requests
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Dataset:
url = "https://www.gutenberg.org/files/11/11-0.txt"
raw_text = requests.get(url).text
raw_text = re.sub(r'\s+', ' ', raw_text)  # Replace multiple whitespace with a single space
raw_text = raw_text.strip()  # Remove leading and trailing whitespace
raw_text = raw_text[:10000]  # Use only the first 10,000 characters for this example
print(len(raw_text), raw_text[:500])

specials = ["<pad>", "<sos>", "<eos>", "<unk>"]
end_punctuations =  [".", "!", "?"] # Common sentence-ending punctuation marks

# Tokenization (with sentence boundaries):
def tokenize_with_sentence_boundaries(text):
    base_tokens = re.findall(r"\w+|[^\w\s*]", text)
    tokens = ["<sos>"]
    for token in base_tokens:
        tokens.append(token)
        if token in end_punctuations:
            tokens.append("<eos>")
            tokens.append("<sos>")
    if tokens and tokens[-1] == "<sos>":
            tokens.pop()
    if tokens and tokens[-1] != "<eos>":
            tokens.append("<eos>")
    return tokens


processed_text = tokenize_with_sentence_boundaries(raw_text) 
print(f"Number of tokens: {len(processed_text)}, First 500 characters: {processed_text[:500]}")

def encoder(tokens):
    vocab = sorted(set(specials + tokens))
    vocab = {token: idx for idx , token in enumerate(vocab)}
    encoded = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    return encoded, vocab


encoded_text, vocab = encoder(processed_text)
print(f"Vocabulary size: {len(vocab)}, \nFirst 100 tokens in vocabulary: {list(vocab.keys())[:100]} , \nFirst 100 encoded tokens: {encoded_text[:100]}")

# Dataset and DataLoader:
class TextDataset(Dataset):
    def __init__(self, encoded_text, seq_length):
        self.encoded_text = encoded_text
        self.seq_length = seq_length

    def __len__(self):
        return len(self.encoded_text) - self.seq_length

    def __getitem__(self, idx):
        input_seq = self.encoded_text[idx:idx + self.seq_length]
        target_seq = self.encoded_text[idx + 1:idx + self.seq_length + 1]
        return torch.tensor(input_seq, dtype=torch.long), torch.tensor(target_seq, dtype=torch.long)
    
dataset = TextDataset(encoded_text, seq_length=50)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Hyperparameters:
embedding_dim = 128
hidden_dim = 256
vocab_size = len(vocab)
model = RNNLanguageModel(vocab_size, embedding_dim, hidden_dim, embedding_layer)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# Embedding Layer:
embedding_layer = nn.Embedding(vocab_size, embedding_dim)



Device: cpu
10000 *** START OF THE PROJECT GUTENBERG EBOOK 11 *** [Illustration] Alice’s Adventures in Wonderland by Lewis Carroll THE MILLENNIUM FULCRUM EDITION 3.0 Contents CHAPTER I. Down the Rabbit-Hole CHAPTER II. The Pool of Tears CHAPTER III. A Caucus-Race and a Long Tale CHAPTER IV. The Rabbit Sends in a Little Bill CHAPTER V. Advice from a Caterpillar CHAPTER VI. Pig and Pepper CHAPTER VII. A Mad Tea-Party CHAPTER VIII. The Queen’s Croquet-Ground CHAPTER IX. The Mock Turtle’s Story CHAPTER X. The Lobster
Number of tokens: 2509, First 500 characters: ['<sos>', 'START', 'OF', 'THE', 'PROJECT', 'GUTENBERG', 'EBOOK', '11', '[', 'Illustration', ']', 'Alice', '’', 's', 'Adventures', 'in', 'Wonderland', 'by', 'Lewis', 'Carroll', 'THE', 'MILLENNIUM', 'FULCRUM', 'EDITION', '3', '.', '<eos>', '<sos>', '0', 'Contents', 'CHAPTER', 'I', '.', '<eos>', '<sos>', 'Down', 'the', 'Rabbit', '-', 'Hole', 'CHAPTER', 'II', '.', '<eos>', '<sos>', 'The', 'Pool', 'of', 'Tears', 'CHAPTER', 'III', '.', '

### 1.a. Basic RNN Model:

In [64]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_layer):
        super().__init__()
        self.embedding = embedding_layer
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out)

model = RNNLanguageModel(vocab_size, embedding_dim, hidden_dim, embedding_layer).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


# Training Loop:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")


Epoch 1/5, Loss: 3.9767
Epoch 2/5, Loss: 1.2686
Epoch 3/5, Loss: 0.3712
Epoch 4/5, Loss: 0.1829
Epoch 5/5, Loss: 0.1242


### 1.b. LSTM Model:

In [65]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_layer):
        super().__init__()
        self.embedding = embedding_layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out)

lstm_model = LSTMLanguageModel(vocab_size, embedding_dim, hidden_dim, embedding_layer).to(device)
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

# Training Loop for LSTM:
for epoch in range(num_epochs):
    lstm_model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        lstm_optimizer.zero_grad()
        outputs = lstm_model(inputs)
        loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))
        loss.backward()
        lstm_optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"LSTM Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

LSTM Epoch 1/5, Loss: 4.6989
LSTM Epoch 2/5, Loss: 2.1799
LSTM Epoch 3/5, Loss: 0.7830
LSTM Epoch 4/5, Loss: 0.3465
LSTM Epoch 5/5, Loss: 0.2169


torch.int64 torch.int64
